In [7]:
# ============================================================
# NB08_Paper_Trader (v2)
# Purpose: A fully automated paper trading simulation.
# Auto-sells losers, auto-buys top candidates, manages fake cash.
# Changes in v2:
#   - Stocks sold this run are NOT re-bought in the same run
#   - Explicit schemas, so an empty portfolio no longer crashes the write
# ============================================================

import pandas as pd
from datetime import datetime
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, DoubleType, LongType, DateType
)

TODAY = datetime.utcnow().date()
MAX_POSITION_SIZE = 50000.0  # Max Rs. 50k per stock
STOP_LOSS_PCT = 0.07         # 7% Stop Loss
MIN_CASH_TO_BUY = 10000.0

PORTFOLIO_COLS = ["symbol", "buy_date", "buy_price", "quantity", "invested_amount"]

PORTFOLIO_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("buy_date", DateType(), True),
    StructField("buy_price", DoubleType(), True),
    StructField("quantity", LongType(), True),
    StructField("invested_amount", DoubleType(), True),
])

CASH_SCHEMA = StructType([
    StructField("account", StringType(), True),
    StructField("cash_balance", DoubleType(), True),
    StructField("last_updated", DateType(), True),
])

LEDGER_SCHEMA = StructType([
    StructField("symbol", StringType(), True),
    StructField("buy_date", DateType(), True),
    StructField("sell_date", DateType(), True),
    StructField("buy_price", DoubleType(), True),
    StructField("sell_price", DoubleType(), True),
    StructField("quantity", LongType(), True),
    StructField("realized_pnl", DoubleType(), True),
    StructField("pnl_pct", DoubleType(), True),
    StructField("reason", StringType(), True),
])


def to_date(value):
    """Convert any date-like value (date, Timestamp, str) to datetime.date."""
    if value is None or (not isinstance(value, str) and pd.isna(value)):
        return None
    return pd.to_datetime(value).date()


# ------------------------------------------------------------
# 1. Load Current State
# ------------------------------------------------------------
portfolio = spark.read.table("gold_paper_portfolio").toPandas()
cash_table = spark.read.table("gold_paper_cash").toPandas()
ledger_rows = []

cash_balance = float(cash_table["cash_balance"].iloc[0])

# Load Market Data
candidates = spark.read.table("gold_stock_candidates").toPandas()
prices = spark.sql(
    "SELECT symbol, close FROM silver_price_features "
    "WHERE trade_date = (SELECT MAX(trade_date) FROM silver_price_features)"
).toPandas()

candidate_symbols = set(candidates["symbol"].tolist()) if len(candidates) > 0 else set()

print(f"💰 Starting Cash: ₹{cash_balance:,.2f}")
print(f"📈 Current Holdings: {len(portfolio)} stocks")

# ------------------------------------------------------------
# PART 1: EVALUATE & SELL (Auto-Exits)
# ------------------------------------------------------------
active_portfolio = []
sold_today = set()   # symbols sold this run -> blocked from re-buy

for idx, row in portfolio.iterrows():
    sym = row["symbol"]
    buy_px = float(row["buy_price"])
    qty = int(row["quantity"])

    # Get current price (fall back to buy price if missing)
    curr_px_row = prices[prices["symbol"] == sym]
    curr_px = float(curr_px_row["close"].iloc[0]) if not curr_px_row.empty else buy_px

    # Check Rules
    stop_loss_px = buy_px * (1 - STOP_LOSS_PCT)
    is_still_candidate = sym in candidate_symbols

    sell = False
    reason = ""

    if curr_px <= stop_loss_px:
        sell = True
        reason = "STOP_LOSS_TRIGGERED"
    elif not is_still_candidate:
        sell = True
        reason = "DROPPED_FROM_CANDIDATES"

    if sell:
        # Execute Sell
        sell_value = curr_px * qty
        realized_pnl = sell_value - (buy_px * qty)
        pnl_pct = (curr_px / buy_px - 1) * 100
        cash_balance += sell_value  # Add cash back to bank
        sold_today.add(sym)

        ledger_rows.append({
            "symbol": sym, "buy_date": to_date(row["buy_date"]), "sell_date": TODAY,
            "buy_price": buy_px, "sell_price": curr_px, "quantity": qty,
            "realized_pnl": float(realized_pnl), "pnl_pct": float(pnl_pct), "reason": reason
        })
        print(f"🔴 SOLD: {sym} at ₹{curr_px} (Reason: {reason}) | PnL: ₹{realized_pnl:.2f}")
    else:
        # Keep holding
        active_portfolio.append(row.to_dict())

portfolio = (
    pd.DataFrame(active_portfolio) if active_portfolio
    else pd.DataFrame(columns=PORTFOLIO_COLS)
)

# ------------------------------------------------------------
# PART 2: EVALUATE & BUY (Auto-Entries)
# ------------------------------------------------------------
# Top candidates we don't already own and did not just sell
current_holdings = portfolio["symbol"].tolist() if not portfolio.empty else []
blocked = set(current_holdings) | sold_today

buy_list = (
    candidates[~candidates["symbol"].isin(blocked)]
    .sort_values("candidate_rank")
)

for idx, row in buy_list.iterrows():
    if cash_balance < MIN_CASH_TO_BUY:
        print("⚠️ Out of cash! Cannot buy more.")
        break

    sym = row["symbol"]
    price = float(row["close"])
    if price <= 0:
        continue

    # Buy maximum allowed position size or whatever cash is left
    invest_amount = min(MAX_POSITION_SIZE, cash_balance)
    qty = int(invest_amount // price)

    if qty > 0:
        actual_cost = qty * price
        cash_balance -= actual_cost

        new_position = {
            "symbol": sym,
            "buy_date": TODAY,
            "buy_price": price,
            "quantity": qty,
            "invested_amount": actual_cost
        }
        portfolio = pd.concat([portfolio, pd.DataFrame([new_position])], ignore_index=True)
        print(f"🟢 BOUGHT: {qty} shares of {sym} at ₹{price} (Cost: ₹{actual_cost:,.2f})")

if sold_today:
    print(f"ℹ️ Not re-bought this run (just sold): {sorted(sold_today)}")

# ------------------------------------------------------------
# PART 3: UPDATE DATABASE
# ------------------------------------------------------------
# 1. Update Portfolio (explicit schema -> works even when empty)
portfolio_records = [
    (
        str(r["symbol"]),
        to_date(r["buy_date"]),
        float(r["buy_price"]),
        int(r["quantity"]),
        float(r["invested_amount"]) if pd.notna(r.get("invested_amount")) else float(r["buy_price"]) * int(r["quantity"]),
    )
    for _, r in portfolio.iterrows()
]
(
    spark.createDataFrame(portfolio_records, schema=PORTFOLIO_SCHEMA)
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_paper_portfolio")
)

# 2. Update Cash
cash_records = [("PAPER_TRADER", float(cash_balance), TODAY)]
(
    spark.createDataFrame(cash_records, schema=CASH_SCHEMA)
    .write.format("delta").mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold_paper_cash")
)

# 3. Append to Ledger
if ledger_rows:
    ledger_records = [
        (
            r["symbol"], r["buy_date"], r["sell_date"], r["buy_price"],
            r["sell_price"], r["quantity"], r["realized_pnl"], r["pnl_pct"], r["reason"]
        )
        for r in ledger_rows
    ]
    (
        spark.createDataFrame(ledger_records, schema=LEDGER_SCHEMA)
        .write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable("gold_paper_ledger")
    )

print(f"\n✅ Paper Trading Complete. Holdings: {len(portfolio)} | Remaining Cash: ₹{cash_balance:,.2f}")

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 3, Finished, Available, Finished, False)

/tmp/ipykernel_7382/2294455548.py:169: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  portfolio = pd.concat([portfolio, pd.DataFrame([new_position])], ignore_index=True)



✅ Paper Trading Complete. Holdings: 15 | Remaining Cash: ₹275,294.80


In [8]:
display(spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv"))

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5bc723b-a498-4ea1-81a9-e466d4dcbb79)

In [9]:
df = spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv")
# df now is a Spark DataFrame containing CSV data from "Files/portfolio/holdings.csv".
display(df)

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 57ef1e8a-8ac2-418a-b104-7ad061ed2790)

In [10]:
display(
    spark.sql("""
        SELECT
            portfolio_action,
            COUNT(*) AS total_positions,
            SUM(investment_cost) AS total_cost,
            SUM(current_value) AS total_value,
            SUM(unrealized_pnl) AS total_pnl
        FROM gold_portfolio_actions
        GROUP BY portfolio_action
    """)
)

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f5cfe1da-1430-4edb-8e39-f1d4a8c84d8a)

In [11]:
display(
    spark.sql("""
        SELECT
            symbol,
            buy_price,
            current_price,
            pnl_pct,
            composite_score,
            eligibility_status,
            portfolio_action,
            action_reason
        FROM gold_portfolio_actions
        ORDER BY
            CASE portfolio_action 
                WHEN 'EXIT_CANDIDATE' THEN 1
                WHEN 'UNDER_REVIEW' THEN 2
                ELSE 3 END,
            pnl_pct ASC
    """)
)

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, f7d05478-dd43-4502-92ca-aa42e60a340e)

In [12]:
import pandas as pd

# 1. Make sure folder exists
mssparkutils.fs.mkdirs("Files/portfolio")

# 2. Create header-only file
empty = pd.DataFrame(columns=["symbol","buy_date","buy_price","quantity","broker"])
empty.to_csv("/lakehouse/default/Files/portfolio/holdings.csv", index=False)

print("Recreated empty holdings.csv")
display(spark.read.format("csv").option("header","true").load("Files/portfolio/holdings.csv"))

StatementMeta(, 1c9e79d6-ad07-44c1-940d-623a14d8fe8d, 8, Finished, Available, Finished, False)

Recreated empty holdings.csv


SynapseWidget(Synapse.DataFrame, 5f8c04e3-4f00-433b-b294-f147808c9d0c)